# Timeline to Intervention Mapping Test

This notebook tests the default mapping bridge:

`TimelineEvent -> mapping profiles -> Intervention objects`

No epidemic simulation is run here.


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "extensions" / "scenario_api").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate extensions/scenario_api from current working directory")

extensions_dir = repo_root / "extensions"
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api import (
    load_timeline_events_from_processed,
    filter_timeline_events,
    assign_relative_day_indices,
    default_mask_mapping_profile,
    default_contact_policy_mapping_profile,
    default_testing_tracing_mapping_profile,
    default_mask_effectiveness_profiles,
    map_timeline_events_to_interventions,
    MaskAdoptionIntervention,
    ContactReductionIntervention,
    TestingIntensityIntervention,
    TracingIntensityIntervention,
    InterventionSet,
    compile_network_multipliers,
)

import pandas as pd

## B) Load processed timeline file

In [ ]:
processed_path = repo_root / "data" / "processed" / "oxcgrt_finland_2020_2022_timeline.csv"

events = load_timeline_events_from_processed(processed_path)
print("Loaded events:", len(events))
print("Date range:", min(e.date for e in events), "->", max(e.date for e in events))

## C) Filter selected event types

In [ ]:
selected_types = {"facial_coverings", "school_closing", "workplace_closing", "testing_policy", "contact_tracing"}
selected_events = [e for e in events if e.event_type in selected_types]
print("Selected events:", len(selected_events))

pd.DataFrame([
    {
        "date": e.date,
        "event_type": e.event_type,
        "value": e.value,
    }
    for e in selected_events[:15]
])

## D) Convert dates to relative days

In [ ]:
reference_start_date = "2020-01-01"
selected_events_with_days = assign_relative_day_indices(selected_events, reference_start_date)

pd.DataFrame([
    {
        "date": e.date,
        "relative_day": e.metadata.get("relative_day"),
        "event_type": e.event_type,
        "value": e.value,
    }
    for e in selected_events_with_days[:15]
])

## E) Create default mapping profiles

In [ ]:
mask_mapping_profile = default_mask_mapping_profile()
contact_mapping_profile = default_contact_policy_mapping_profile()
testing_tracing_mapping_profile = default_testing_tracing_mapping_profile()
mask_profiles = default_mask_effectiveness_profiles()

mask_mapping_profile, contact_mapping_profile.name, testing_tracing_mapping_profile.name

## F) Map events to interventions

In [ ]:
mapped_interventions = map_timeline_events_to_interventions(
    selected_events,
    mask_mapping_profile=mask_mapping_profile,
    contact_mapping_profile=contact_mapping_profile,
    testing_tracing_mapping_profile=testing_tracing_mapping_profile,
    mask_profiles=mask_profiles,
    reference_start_date=reference_start_date,
)

print("Mapped interventions:", len(mapped_interventions))

pd.DataFrame([
    {
        "type": type(i).__name__,
        "name": i.name,
        "start": i.start,
        "end": i.end,
        "details": getattr(i, "multipliers", None) or getattr(i, "network_mix", None) or getattr(i, "config", None),
    }
    for i in mapped_interventions[:20]
])

## G) Inspect facial_coverings mapping

In [ ]:
facial_mapped = [i for i in mapped_interventions if isinstance(i, MaskAdoptionIntervention)]
print("Facial-covering mapped interventions:", len(facial_mapped))

pd.DataFrame([
    {
        "name": i.name,
        "start": i.start,
        "end": i.end,
        "work_mix": i.network_mix.get("work"),
        "random_mix": i.network_mix.get("random"),
    }
    for i in facial_mapped[:15]
])

## H) Inspect school/work contact-policy mapping

In [ ]:
contact_mapped = [i for i in mapped_interventions if isinstance(i, ContactReductionIntervention)]
print("Contact reduction mapped interventions:", len(contact_mapped))

pd.DataFrame([
    {
        "name": i.name,
        "start": i.start,
        "end": i.end,
        "multipliers": i.multipliers,
    }
    for i in contact_mapped[:20]
])

## I) Simple summary

In [ ]:
summary = pd.Series([type(i).__name__ for i in mapped_interventions]).value_counts().rename_axis("intervention_type").reset_index(name="count")
display(summary)

selected_day = 100
mult = compile_network_multipliers(InterventionSet(mapped_interventions), t=selected_day)
print(f"Compiled network multipliers at day {selected_day}:", mult)

testing_n = sum(isinstance(i, TestingIntensityIntervention) for i in mapped_interventions)
tracing_n = sum(isinstance(i, TracingIntensityIntervention) for i in mapped_interventions)
print("Testing placeholder interventions:", testing_n)
print("Tracing placeholder interventions:", tracing_n)